# RiverSentinel — Preprocessing Pipeline (Stage 1)

**Purpose.** Produces the vector and raster inputs required by the
modelling stage: a legally-defined riparian buffer polygon and a clipped,
tiled satellite composite, for each configured region.

**Inputs.** OpenStreetMap waterway data (Geofabrik, country extract);
Sentinel-2 surface-reflectance imagery (Earth Engine).

**Outputs (per region `{name}`), consumed by Stage 2:**
| File | Description |
|---|---|
| `data/processed/{name}_60m_riparian_zone.geojson` | 60m buffer polygon around river/stream centerlines |
| `data/processed/{name}_composite_clipped.tif` | Sentinel-2 composite, clipped to the buffer |
| `data/processed/{name}_composite_clipped_8bit.tif` | Same, rescaled to 8-bit for pixel-level analysis |
| `data/tiles/{name}/tile_*.png` | Clipped composite, cut into fixed-size tiles |

**Preconditions.** A Google Cloud project registered for Earth Engine
access, and network access to Geofabrik and the Earth Engine API. See
`requirements1.txt` for the pinned dependency set.


## Repository hygiene

Generated data and model artifacts are excluded from version control.
Raw and processed geospatial files are regenerated by re-running this
pipeline; only source code and pinned dependencies are committed.

In [1]:
import os

gitignore_contents = """
# Raw + processed geospatial data -- regenerated by running this pipeline
data/raw/
data/processed/
data/vectors/
data/tiles/

# Trained model artifacts -- regenerated by re-running the training cells
models/
runs/

# Standard Python / notebook noise
__pycache__/
*.pyc
.ipynb_checkpoints/

# Local virtualenv
.venv/
"""

with open("../.gitignore", "w") as f:
    f.write(gitignore_contents.strip() + "\n")

print("Wrote .gitignore")

Wrote .gitignore


## 1.1 Configuration

Study regions are defined as `{name, bbox}` pairs. Each downstream step
iterates over `REGIONS`, so adding a region requires only a new bounding
box entry -- no changes to processing logic. Bounding boxes below were
derived by querying the actual downloaded OSM waterway data for named
river corridors (see Stage 1 region-selection notes in project
documentation); they are not estimates.

In [2]:
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import box, LineString

DIRS = ["../data/raw", "../data/processed", "../data/vectors", "../data/tiles", "../models", "../runs"]
for d in DIRS:
    os.makedirs(d, exist_ok=True)

REGIONS = [
    {"name": "kasarani",   "bbox": (36.80, -1.32, 36.95, -1.20)},
    {"name": "gatharaini", "bbox": (36.8984, -1.2522, 37.0221, -1.1930)},
    {"name": "motoine",    "bbox": (36.6702, -1.3346, 36.8105, -1.2803)},
]

print(f"{len(REGIONS)} region(s) configured: {[r['name'] for r in REGIONS]}")

3 region(s) configured: ['kasarani', 'gatharaini', 'motoine']


## 1.2 Data Acquisition — Waterways

Source: Geofabrik OpenStreetMap extract
(`https://download.geofabrik.de/africa/kenya-latest-free.shp.zip`).
Geofabrik does not provide a waterways-only extract; the archive contains
all OSM layers for the country and is approximately 936MB for Kenya. It is
downloaded once into `data/raw/` (gitignored) and reused on subsequent
runs.

In [3]:
import requests, zipfile, io

GEOFABRIK_URL = "https://download.geofabrik.de/africa/kenya-latest-free.shp.zip"
RAW_DIR = "../data/raw/kenya_osm"

def download_and_unzip(url, target_dir):
    if os.path.exists(target_dir) and os.listdir(target_dir):
        print(f"Already downloaded: {target_dir}")
        return
    os.makedirs(target_dir, exist_ok=True)
    print(f"Downloading {url} ...")
    response = requests.get(url, timeout=120)
    response.raise_for_status()
    with zipfile.ZipFile(io.BytesIO(response.content)) as zf:
        zf.extractall(target_dir)
    print(f"Extracted into {target_dir}")

download_and_unzip(GEOFABRIK_URL, RAW_DIR)
VECTOR_INPUT = os.path.join(RAW_DIR, "gis_osm_waterways_free_1.shp")

Already downloaded: ../data/raw/kenya_osm


## 1.3 Vector Preprocessing — Riparian Buffer

River/stream centerlines are buffered by a fixed legal distance (60m).
`EPSG:4326` measures distance in degrees, which is not uniform with
latitude; buffering is performed in `EPSG:32737` (UTM Zone 37S, a
projected CRS covering this region in meters) and the result is
reprojected back to `EPSG:4326` for downstream consumers.

In [4]:
def execute_vector_preprocessing(input_shapefile, region_bbox, output_vector_path, buffer_meters=60):
    if not os.path.exists(input_shapefile):
        raise FileNotFoundError(f"{input_shapefile} not found -- run Section 1.2 first.")

    raw_gdf = gpd.read_file(input_shapefile, bbox=box(*region_bbox))

    if "fclass" in raw_gdf.columns:
        filtered_rivers = raw_gdf[raw_gdf["fclass"].isin(["river", "stream"])].copy()
    else:
        filtered_rivers = raw_gdf.copy()

    rivers_metric = filtered_rivers.to_crs(epsg=32737)
    buffer_metric = rivers_metric.buffer(buffer_meters)
    buffer_gdf = gpd.GeoDataFrame(geometry=buffer_metric, crs="EPSG:32737")
    buffer_global = buffer_gdf.to_crs(epsg=4326)

    buffer_global.to_file(output_vector_path, driver="GeoJSON")
    return buffer_global, len(filtered_rivers)

for region in REGIONS:
    name = region["name"]
    out_path = f"../data/processed/{name}_60m_riparian_zone.geojson"
    boundary, n_features = execute_vector_preprocessing(VECTOR_INPUT, region["bbox"], out_path)
    region["vector_output"] = out_path
    print(f"[{name}] {n_features} river/stream feature(s) -> {out_path}")

[kasarani] 188 river/stream feature(s) -> ../data/processed/kasarani_60m_riparian_zone.geojson


[gatharaini] 101 river/stream feature(s) -> ../data/processed/gatharaini_60m_riparian_zone.geojson


[motoine] 85 river/stream feature(s) -> ../data/processed/motoine_60m_riparian_zone.geojson


## 1.4 Data Acquisition — Satellite Composite

A cloud-filtered Sentinel-2 surface-reflectance composite is retrieved per
region via Earth Engine. The acquisition window is a rolling 90-day period
ending at execution time, ranked by cloud cover, rather than a fixed
historical range -- output reflects current conditions on every run.

**Constraints.**
- `ee.Initialize()` requires an explicit Google Cloud project
  (`EE_PROJECT_ID`); anonymous/project-less initialization is not
  supported by the current API version.
- Earth Engine's synchronous download path (`getDownloadURL`) caps a
  single request at 48MB of pixel data. At native 10m resolution this
  exceeds that limit for these AOIs; output scale is set to 15m. Larger
  areas or finer resolution require `ee.batch.Export.image.toDrive` /
  `toCloudStorage` instead, which is not subject to this cap.

In [5]:
import os
import datetime
import ee
import geemap

ee.Authenticate()

EE_PROJECT_ID = os.environ.get("EE_PROJECT_ID", "solar-haven-349708")
ee.Initialize(project=EE_PROJECT_ID)

RECENT_DAYS = 90

def pull_recent_composite(region_bbox, out_path, scale=15):
    aoi = ee.Geometry.BBox(*region_bbox)
    end_date = ee.Date(datetime.date.today().isoformat())
    start_date = end_date.advance(-RECENT_DAYS, "day")

    recent_collection = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(aoi)
        .filterDate(start_date, end_date)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 20))
        .sort("CLOUDY_PIXEL_PERCENTAGE")
    )
    n_scenes = recent_collection.size().getInfo()
    if n_scenes == 0:
        raise RuntimeError(
            f"No scenes under 20% cloud cover in the last {RECENT_DAYS} days for this region."
        )

    composite = recent_collection.median().select(["B4", "B3", "B2"]).clip(aoi)
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    geemap.ee_export_image(composite, filename=out_path, scale=scale, region=aoi)
    return n_scenes

for region in REGIONS:
    name = region["name"]
    out_path = f"../data/raw/recent_composite/{name}_composite.tif"
    n_scenes = pull_recent_composite(region["bbox"], out_path)
    region["composite_path"] = out_path
    print(f"[{name}] {n_scenes} scene(s) composited -> {out_path}")

Generating URL ...


Please wait ...


Data downloaded to /home/miringu/Downloads/Group5/data/raw/recent_composite/kasarani_composite.tif
[kasarani] 6 scene(s) composited -> ../data/raw/recent_composite/kasarani_composite.tif


Generating URL ...


Please wait ...


Data downloaded to /home/miringu/Downloads/Group5/data/raw/recent_composite/gatharaini_composite.tif
[gatharaini] 6 scene(s) composited -> ../data/raw/recent_composite/gatharaini_composite.tif


Generating URL ...


Please wait ...


Data downloaded to /home/miringu/Downloads/Group5/data/raw/recent_composite/motoine_composite.tif
[motoine] 14 scene(s) composited -> ../data/raw/recent_composite/motoine_composite.tif


## 1.5 Repository Size Validation

GitHub issues a soft warning at 50MB per file and blocks pushes above
100MB. Raster outputs are checked against these thresholds at the point of
creation, rather than at commit time.

In [6]:
def check_github_size(path, warn_mb=50, block_mb=100):
    size_mb = os.path.getsize(path) / (1024 * 1024)
    if size_mb >= block_mb:
        print(f"BLOCKED: {path} is {size_mb:.1f}MB -- exceeds GitHub's 100MB push limit.")
    elif size_mb >= warn_mb:
        print(f"WARNING: {path} is {size_mb:.1f}MB -- under the push limit; consider Git LFS if versioning is required.")
    else:
        print(f"OK: {path} is {size_mb:.1f}MB.")
    return size_mb

for region in REGIONS:
    path = region.get("composite_path")
    if path and os.path.exists(path):
        check_github_size(path)

OK: ../data/raw/recent_composite/kasarani_composite.tif is 6.1MB.
OK: ../data/raw/recent_composite/gatharaini_composite.tif is 2.5MB.
OK: ../data/raw/recent_composite/motoine_composite.tif is 2.6MB.


## 1.6 Raster Preprocessing — Clip, Tile, Normalize

Each composite is clipped to its region's buffer polygon (restricting
downstream processing to the legally relevant area) and cut into fixed
640px tiles.

Sentinel-2 surface-reflectance values are unscaled and typically range
0-13,000+, not the 0-255 range assumed by standard 8-bit image processing.
Pixel-level operations in Stage 2 (thresholding, GLCM texture) require the
0-255 range; a rescaled copy is produced here (reflectance clipped to
[0, 3000], mapped to [0, 255]) so Stage 2 can consume it directly.

In [7]:
def clip_and_tile_satellite_composite(source_raster_path, riparian_buffer_geojson_path,
                                       clipped_output_path, tile_output_dir, tile_size=640):
    import rasterio
    from rasterio.mask import mask
    from PIL import Image

    buffer_gdf = gpd.read_file(riparian_buffer_geojson_path)

    with rasterio.open(source_raster_path) as src:
        if buffer_gdf.crs != src.crs:
            buffer_gdf = buffer_gdf.to_crs(src.crs)
        geoms = [g.__geo_interface__ for g in buffer_gdf.geometry]
        clipped_array, clipped_transform = mask(src, geoms, crop=True)
        clipped_meta = src.meta.copy()
        clipped_meta.update({
            "height": clipped_array.shape[1],
            "width": clipped_array.shape[2],
            "transform": clipped_transform,
        })

    with rasterio.open(clipped_output_path, "w", **clipped_meta) as dst:
        dst.write(clipped_array)

    os.makedirs(tile_output_dir, exist_ok=True)
    bands, height, width = clipped_array.shape
    rgb = clipped_array[:3] if bands >= 3 else clipped_array
    tile_count = 0
    for row in range(0, height, tile_size):
        for col in range(0, width, tile_size):
            tile = rgb[:, row:row + tile_size, col:col + tile_size]
            if tile.shape[1] < 10 or tile.shape[2] < 10:
                continue
            tile_img = np.moveaxis(tile, 0, -1)
            tile_img = np.clip(tile_img, 0, 255).astype("uint8")
            Image.fromarray(tile_img).save(os.path.join(tile_output_dir, f"tile_{tile_count:04d}.png"))
            tile_count += 1
    return clipped_output_path, tile_count


def rescale_to_8bit(input_path, output_path, reflectance_max=3000):
    import rasterio
    with rasterio.open(input_path) as src:
        arr = src.read().astype(np.float32)
        meta = src.meta.copy()
    stretched = np.clip(arr / reflectance_max * 255.0, 0, 255).astype(np.uint8)
    meta.update(dtype="uint8")
    with rasterio.open(output_path, "w", **meta) as dst:
        dst.write(stretched)
    return output_path


for region in REGIONS:
    name = region["name"]
    composite_path = region.get("composite_path")
    if not composite_path or not os.path.exists(composite_path):
        print(f"[{name}] skipped -- no composite available")
        continue

    clipped_path = f"../data/processed/{name}_composite_clipped.tif"
    tile_dir = f"../data/tiles/{name}"
    _, tile_count = clip_and_tile_satellite_composite(composite_path, region["vector_output"], clipped_path, tile_dir)
    region["clipped_path"] = clipped_path
    region["tile_dir"] = tile_dir
    region["tile_count"] = tile_count

    stretched_path = f"../data/processed/{name}_composite_clipped_8bit.tif"
    rescale_to_8bit(clipped_path, stretched_path)
    region["stretched_path"] = stretched_path

    print(f"[{name}] clipped -> {clipped_path} ({tile_count} tiles); 8-bit -> {stretched_path}")

[kasarani] clipped -> ../data/processed/kasarani_composite_clipped.tif (4 tiles); 8-bit -> ../data/processed/kasarani_composite_clipped_8bit.tif


[gatharaini] clipped -> ../data/processed/gatharaini_composite_clipped.tif (2 tiles); 8-bit -> ../data/processed/gatharaini_composite_clipped_8bit.tif


[motoine] clipped -> ../data/processed/motoine_composite_clipped.tif (2 tiles); 8-bit -> ../data/processed/motoine_composite_clipped_8bit.tif


## 1.7 Stage Output Manifest

Validates that every region produced the files Stage 2 requires before
this notebook is considered complete.

In [8]:
required = ["vector_output", "clipped_path", "stretched_path", "tile_dir"]
manifest = []
all_ok = True
for region in REGIONS:
    name = region["name"]
    missing = [k for k in required if not region.get(k) or not os.path.exists(region[k])]
    status = "OK" if not missing else f"INCOMPLETE ({missing})"
    all_ok = all_ok and not missing
    manifest.append({"region": name, "status": status, **{k: region.get(k) for k in required}})

manifest_df = pd.DataFrame(manifest)
print("Stage 1 complete -- all regions ready for Stage 2" if all_ok else "Stage 1 INCOMPLETE -- see status column")
manifest_df

Stage 1 complete -- all regions ready for Stage 2


,region,status,vector_output,clipped_path,stretched_path,tile_dir
0,kasarani,OK,../data/processed/kasarani_60m_riparian_zone.g...,../data/processed/kasarani_composite_clipped.tif,../data/processed/kasarani_composite_clipped_8...,../data/tiles/kasarani
1,gatharaini,OK,../data/processed/gatharaini_60m_riparian_zone...,../data/processed/gatharaini_composite_clipped...,../data/processed/gatharaini_composite_clipped...,../data/tiles/gatharaini
2,motoine,OK,../data/processed/motoine_60m_riparian_zone.ge...,../data/processed/motoine_composite_clipped.tif,../data/processed/motoine_composite_clipped_8b...,../data/tiles/motoine
